# How RAG indexing works in this repo

This notebook answers two questions:

1. **What do we index?** — how each source dataset becomes `data/corpus_*/*.txt`
2. **How does each method index?** — what `build_index()` writes, and how query-time retrieval uses it

Every bake-off method eventually reads the same on-disk shape: a directory of `# Title\n\nbody` text files. Methods differ in what they *build* from those files (vector chunks, entity graphs, trees, propositions).

| Dataset notebook | Indexed corpus | Eval QA |
|------------------|----------------|---------|
| `hotpot_metric_autopsy.ipynb` / `rag_benchmark_executed.ipynb` | `data/corpus_hotpot/` | `data/qa/hotpot_eval.json` |
| `rag_benchmark.ipynb` | `data/corpus_graphrag_bench/` | `data/qa/graphrag_bench_eval.json` |
| `multihop_rag_bench.ipynb` | `data/corpus_multihop/` | `data/qa/multihop_eval.json` |


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from rag_benchmark.charts import METHOD_LABELS
from rag_benchmark.corpus import chunk_documents, load_documents

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 40)

CORPORA = {
    "HotpotQA": ROOT / "data" / "corpus_hotpot",
    "GraphRAG-Bench": ROOT / "data" / "corpus_graphrag_bench",
    "MultiHop-RAG": ROOT / "data" / "corpus_multihop",
}
META = {
    name: json.loads((ROOT / "data" / "qa" / f"{key}_meta.json").read_text())
    for name, key in [
        ("HotpotQA", "hotpot"),
        ("GraphRAG-Bench", "graphrag_bench"),
        ("MultiHop-RAG", "multihop"),
    ]
}

rows = []
for name, path in CORPORA.items():
    m = META[name]
    rows.append(
        {
            "dataset": name,
            "corpus_dir": str(path.relative_to(ROOT)),
            "n_docs_on_disk": len(list(path.glob("*.txt"))) if path.exists() else 0,
            "n_questions": m.get("n_questions"),
            "source": m.get("source") or m.get("citation") or m.get("paper"),
            "setting": m.get("setting") or m.get("subset") or "—",
        }
    )
display(Markdown("### Indexed corpora in this checkout"))
display(pd.DataFrame(rows))

### Indexed corpora in this checkout

,dataset,corpus_dir,n_docs_on_disk,n_questions,source,setting
0,HotpotQA,data/corpus_hotpot,231,24,hotpotqa/hotpot_qa (HuggingFace),distractor
1,GraphRAG-Bench,data/corpus_graphrag_bench,27,7,Novel-4128,novel
2,MultiHop-RAG,data/corpus_multihop,30,9,https://arxiv.org/abs/2401.15391,—


## 1. Source datasets → on-disk corpus

Upstream formats differ. The loaders flatten them into the same local contract so every RAG method can share one `load_documents()` path.

```
HF / GitHub source
        │
        ▼
build_*_subset()          # hotpotqa.py / graphrag_bench.py / multihop_rag.py
        │
        ├── data/corpus_*/{slug}.txt     # # Title + body
        └── data/qa/*_eval.json          # id, question, expected_answer, type…
                │
                ▼
        load_documents(corpus_dir)       # corpus.py → Document(doc_id, title, text)
                │
                ▼
        method.build_index()             # Chroma / GraphRAG parquet / LightRAG / …
```


### HotpotQA (distractor)

| | |
|--|--|
| **Paper** | Yang et al., EMNLP 2018 |
| **HF** | `hotpotqa/hotpot_qa`, config `distractor`, split `validation` |
| **Loader** | `src/rag_benchmark/hotpotqa.py` → `build_hotpot_subset()` |
| **Corpus** | `data/corpus_hotpot/{wiki_slug}.txt` |
| **QA** | `data/qa/hotpot_eval.json` |

**Source structure (HuggingFace row):** each example has `question`, `answer`, `type` (`bridge` / `comparison`), `level`, and `context` with parallel `title[]` + `sentences[][]` — **2 gold Wikipedia paragraphs + 8 distractors**.

**What we write:** one `.txt` per unique Wikipedia title across the selected questions:

```text
# Animorphs

Animorphs is a science fantasy series of young adult books…
```

Bridge → `query_type=hybrid`; comparison → `query_type=local`. Current subset: hard examples, balanced bridge/comparison.


In [2]:
hotpot_qa = json.loads((ROOT / "data" / "qa" / "hotpot_eval.json").read_text())
hotpot_dir = CORPORA["HotpotQA"]
sample = sorted(hotpot_dir.glob("*.txt"))[:3]

display(Markdown("#### Eval item fields"))
display(pd.DataFrame(hotpot_qa).head(3)[
    ["id", "hotpot_type", "query_type", "question", "expected_answer"]
])

display(Markdown("#### Corpus file sample"))
for p in sample:
    text = p.read_text(encoding="utf-8")
    display(Markdown(f"**`{p.name}`** ({len(text)} chars)"))
    print(text[:500] + ("…" if len(text) > 500 else ""))
    print()

#### Eval item fields

,id,hotpot_type,query_type,question,expected_answer
0,5a8c7595554299585d9e36b6,bridge,hybrid,What government position was held by the woman who portrayed Corliss Archer in the fil...,Chief of Protocol
1,5a85ea095542994775f606a8,bridge,hybrid,"What science fantasy young adult series, told in first person, has a set of companion ...",Animorphs
2,5a8e3ea95542995a26add48d,bridge,hybrid,"The director of the romantic comedy ""Big Stone Gap"" is based in what New York city?","Greenwich Village, New York City"


#### Corpus file sample

**`1995_96_manchester_united_f_c_season.txt`** (657 chars)

# 1995–96 Manchester United F.C. season

The 1995–96 season was Manchester United's fourth season in the Premier League, and their 21st consecutive season in the top division of English football.  United finished the season by becoming the first English team to win the Double (league title and FA Cup) twice.  Their triumph was made all the more remarkable by the fact that Alex Ferguson had sold experienced players Paul Ince, Mark Hughes and Andrei Kanchelskis before the start of the season, and …



**`1996_fa_charity_shield.txt`** (619 chars)

# 1996 FA Charity Shield

The 1996 FA Charity Shield (also known as the Littlewoods FA Charity Shield for sponsorship reasons) was the 74th FA Charity Shield, an annual football match played between the winners of the previous season's Premier League and FA Cup competitions.  The match was played on 11 August 1996 at Wembley Stadium and contested by Manchester United, who had won the Double of Premier League and FA Cup in 1995–96, and Newcastle United, who had finished as runners-up in the Premi…



**`2006_07_qmjhl_season.txt`** (493 chars)

# 2006–07 QMJHL season

The 2006–07 QMJHL season was the 38th season in the history of the Quebec Major Junior Hockey League.  The regular season ran from September 14, 2006 to March 18, 2007.  Eighteen teams played 70 games each in the schedule.  The Lewiston Maineiacs finished first overall in the regular season winning their first Jean Rougeau Trophy.  Lewiston won 16 playoff games, losing only one, en route to their first President's Cup, defeating the Val-d'Or Foreurs in the finals.




### GraphRAG-Bench (Novel split)

| | |
|--|--|
| **Paper** | Xiang et al., ICLR 2026 — *When to use Graphs in RAG* |
| **Source** | Official Novel corpus (`Novel-4128` = Samuel Pepys diary) |
| **Loader** | `src/rag_benchmark/graphrag_bench.py` → `build_graphrag_bench_subset()` |
| **Corpus** | `data/corpus_graphrag_bench/novel_4128_{iii}.txt` |
| **QA** | `data/qa/graphrag_bench_eval.json` |

**Source structure:** one long novel text + questions tagged by task level:
`Fact Retrieval` → `Complex Reasoning` → `Contextual Summarize` → `Creative Generation`.

**What we write:** the novel is split into ~3.5k-character parts so GraphRAG/vector chunkers can index a book-scale document without one giant file:

```text
# Novel-4128 part 0

Produced by David Widger THE DIARY OF SAMUEL PEPYS…
```


In [3]:
gb_qa = json.loads((ROOT / "data" / "qa" / "graphrag_bench_eval.json").read_text())
gb_dir = CORPORA["GraphRAG-Bench"]
display(Markdown("#### Task-level questions"))
display(pd.DataFrame(gb_qa)[["id", "graphrag_bench_type", "query_type", "question", "expected_answer"]].head(8))
parts = sorted(gb_dir.glob("*.txt"))
display(Markdown(f"#### Corpus parts: **{len(parts)}** files"))
display(pd.DataFrame(
    [{"file": p.name, "chars": p.stat().st_size, "title_line": p.read_text(encoding="utf-8").splitlines()[0]}
     for p in parts]
))

#### Task-level questions

,id,graphrag_bench_type,query_type,question,expected_answer
0,Novel-1d62116d,Fact Retrieval,local,"What agreement did Mr. Pepys make with Mr. Goodgroome for singing lessons, according t...",Mr. Pepys agreed to give Mr. Goodgroome 20s. entrance and 20s. a month to teach him to...
1,Novel-a8bad1cf,Fact Retrieval,local,"Where did Lady Sandwich treat Samuel Pepys kindly during his visit, as described in hi...",Lady Sandwich treated Samuel Pepys kindly at the Wardrobe.
2,Novel-95217448,Complex Reasoning,hybrid,How does the King's relationship with Mrs. Palmer connect back to the Portuguese Embas...,"The Portuguese Embassador took leave of the King, who has mistress Mrs. Palmer."
3,Novel-a0c1c9cd,Complex Reasoning,hybrid,"What evidence from the narrative indicates that members of the royal family, including...","The Opera was attended by the King, the Duke, the Duchess, and the Queen of Bohemia."
4,Novel-feba1552,Contextual Summarize,hybrid,"How does the sequence of events involving my Lord's appointment as Embassador, his int...","My Lord Embassador, after being made by the King, ordered Mr. Creed, was angry with W...."
5,Novel-710f8505,Contextual Summarize,hybrid,What does Madame Palmer's presence at the Theatre suggest about the King's interests i...,"The King's mistress, Madame Palmer, was at the Theatre where Harry the 4th was shown, ..."
6,Novel-0d6bcca0,Creative Generation,hybrid,"Rewrite the events as a diary entry from Samuel Pepys, focusing on his preparations fo...","Diary Entry — This day, my chief occupation was in making ready sundry items to be dis..."


#### Corpus parts: **27** files

,file,chars,title_line
0,novel_4128_000.txt,3522,# Novel-4128 part 0
1,novel_4128_001.txt,3522,# Novel-4128 part 1
2,novel_4128_002.txt,3522,# Novel-4128 part 2
3,novel_4128_003.txt,3522,# Novel-4128 part 3
4,novel_4128_004.txt,3522,# Novel-4128 part 4
5,novel_4128_005.txt,3522,# Novel-4128 part 5
6,novel_4128_006.txt,3522,# Novel-4128 part 6
7,novel_4128_007.txt,3522,# Novel-4128 part 7
8,novel_4128_008.txt,3522,# Novel-4128 part 8
9,novel_4128_009.txt,3522,# Novel-4128 part 9


### MultiHop-RAG

| | |
|--|--|
| **Paper** | Tang & Yang, COLM 2024 |
| **HF** | `yixuantt/MultiHopRAG` (questions + `corpus`) |
| **Loader** | `src/rag_benchmark/multihop_rag.py` → `build_multihop_rag_subset()` |
| **Corpus** | `data/corpus_multihop/{iii}_{slug}.txt` |
| **QA** | `data/qa/multihop_eval.json` |

**Source structure:** news articles with titles; each question lists evidence titles spanning **2–4 docs**. Query types: `inference_query`, `comparison_query`, `temporal_query`.

**What we write:** closed-world mini corpus = gold evidence articles + distractors:

```text
# Best sportsbook bonus offers for NFL…
Source: Sporting News

Monday Night Football closes out…
```


In [4]:
mh_qa = json.loads((ROOT / "data" / "qa" / "multihop_eval.json").read_text())
mh_dir = CORPORA["MultiHop-RAG"]
display(Markdown("#### Eval + evidence titles"))
cols = [c for c in ["id", "multihop_type", "query_type", "question", "expected_answer", "evidence_titles"] if c in mh_qa[0]]
display(pd.DataFrame(mh_qa)[cols].head(6))
sample = sorted(mh_dir.glob("*.txt"))[0]
display(Markdown(f"#### Sample corpus file `{sample.name}`"))
print(sample.read_text(encoding="utf-8")[:700])

#### Eval + evidence titles

,id,multihop_type,query_type,question,expected_answer,evidence_titles
0,mh-inf-000-meta,inference_query,hybrid,"What company, recently scrutinized by European consumer groups for its ad-free subscri...",Meta,[European consumer groups band together to fight Meta’s self-serving ad-free sub — bra...
1,mh-inf-001-meta,inference_query,hybrid,"Which company, featured in TechCrunch articles, is both seeking to involve parents in ...",Meta,"[Meta seeks legislation that would require parents to approve teens’ app downloads, Me..."
2,mh-inf-002-uber,inference_query,hybrid,"What company, according to TechCrunch, experienced a 38% decrease in reported sexual a...",Uber,"[Uber sexual assault survivors call for in-car cameras, tech upgrades, Uber sexual ass..."
3,mh-com-003-no,comparison_query,local,Does 'The Independent - Life and Style' article suggest that Taylor Swift is secretive...,no,[Travis Kelce faces backlash after comments about ‘finding a breeder’ resurface amid T...
4,mh-com-004-no,comparison_query,local,Did the TechCrunch article indicate that Google's release of the Gemini Pro model was ...,no,"[Google fakes an AI demo, Grand Theft Auto VI goes viral and Spotify cuts jobs, Is Goo..."
5,mh-com-005-no,comparison_query,local,Does 'The Verge' article suggest that Sam Bankman-Fried set withdrawal permissions bas...,no,"[In the end, the FTX trial was about the friends screwed along the way, Is Sam Bankman..."


#### Sample corpus file `000_best_sportsbook_bonus_offers_for_nfl_monday_night_football_eagles_vs_seahawks_cl.txt`

# Best sportsbook bonus offers for NFL Monday Night Football Eagles vs. Seahawks: Claim over $5,000 in bonuses from Bet365, BetMGM, BetRivers, Caesars Sportsbook, DraftKings and FanDuel 
Source: Sporting News

Monday Night Football closes out the NFL’s Week 15 tonight with a pair of teams desperately looking to snap losing streaks. The Eagles vs. Seahawks matchup should be an exciting game, making it a great time to use our sportsbook bonus codes and links to sign up for new sports betting accounts and claim some fantastic welcome offers.

Before tonight’s MNF kickoff in Seattle, just use our exclusive bonus codes at Bet365, BetMGM, BetRivers, Caesars Sportsbook, DraftKings and FanDuel to cl


## 2. Shared document / chunk layer

`src/rag_benchmark/corpus.py` is the choke point:

| Type | Fields |
|------|--------|
| `Document` | `doc_id` (= filename stem), `title`, `text`, `source_path` |
| `TextChunk` | `chunk_id="{doc_id}__{i}"`, `doc_id`, `text`, `source_path` |

Chunking uses tiktoken `cl100k_base`. Config default (`config/benchmark.yaml`): **chunk_size=600**, **overlap=80** (function defaults are 800/100).

Vector methods index chunks. GraphRAG copies whole `.txt` files into its `input/` and does its own text-unit split. LightRAG / HippoRAG ingest full documents.


In [5]:
docs = load_documents(CORPORA["HotpotQA"], max_documents=5)
chunks = chunk_documents(docs, chunk_size=600, chunk_overlap=80)
display(Markdown(f"Loaded **{len(docs)}** docs → **{len(chunks)}** chunks (size=600, overlap=80)"))
display(pd.DataFrame(
    [{"doc_id": d.doc_id, "chars": len(d.text), "title": d.title} for d in docs]
))
display(pd.DataFrame(
    [{"chunk_id": c.chunk_id, "doc_id": c.doc_id, "chars": len(c.text)} for c in chunks[:8]]
))
display(Markdown("#### First chunk preview"))
print(chunks[0].text[:400] if chunks else "(empty)")

Loaded **5** docs → **5** chunks (size=600, overlap=80)

,doc_id,chars,title
0,1995_96_manchester_united_f_c_season,656,1995 96 manchester united f c season
1,1996_fa_charity_shield,618,1996 fa charity shield
2,2006_07_qmjhl_season,492,2006 07 qmjhl season
3,2007_memorial_cup,963,2007 memorial cup
4,2011_12_qmjhl_season,774,2011 12 qmjhl season


,chunk_id,doc_id,chars
0,1995_96_manchester_united_f_c_season__0,1995_96_manchester_united_f_c_season,656
1,1996_fa_charity_shield__0,1996_fa_charity_shield,618
2,2006_07_qmjhl_season__0,2006_07_qmjhl_season,492
3,2007_memorial_cup__0,2007_memorial_cup,963
4,2011_12_qmjhl_season__0,2011_12_qmjhl_season,774


#### First chunk preview

# 1995–96 Manchester United F.C. season

The 1995–96 season was Manchester United's fourth season in the Premier League, and their 21st consecutive season in the top division of English football.  United finished the season by becoming the first English team to win the Double (league title and FA Cup) twice.  Their triumph was made all the more remarkable by the fact that Alex Ferguson had sold ex


## 3. Method map — what each `build_index()` produces

Labels from `METHOD_LABELS` in `src/rag_benchmark/charts.py`.

| Method key | Label | Index artifact family | Query uses |
|------------|-------|----------------------|------------|
| `semantic_rag` | Semantic (vector) | Chroma lineage collection | dense top-k chunks |
| `rerank_semantic` | Vector + rerank | same Chroma + wider recall | CrossEncoder re-rank |
| `hybrid_dense_sparse` | BM25+dense (RRF) | Chroma + in-memory BM25 | RRF fusion |
| `graph_rag` | GraphRAG global | GraphRAG workspace parquet | `query --method global` |
| `graph_local_rag` | GraphRAG local | **same workspace** | `query --method local` |
| `drift_rag` | DRIFT | same workspace | `query --method drift` |
| `lazygraph_rag` | GraphRAG fast/basic | **separate** lazy workspace | `query --method basic` |
| `hybrid_rag` | Hybrid (vec+graph local) | Chroma + GraphRAG workspace | fuse chunks + local graph |
| `adaptive_rag` | Adaptive router | builds semantic + hybrid | rule route → one of them |
| `frontier_rag` | FrontierRAG | dense/sparse + hybrid | grade → escalate to hybrid |
| `light_rag` | LightRAG (HKUDS) | LightRAG kv/vdb/graphml | mode=`hybrid` (default) |
| `hippo_rag` | HippoRAG 2 | OpenIE + graph.pickle | `rag_qa` |
| `raptor_rag` | RAPTOR | tree.json + Chroma nodes | collapsed-tree retrieve |
| `parent_child_rag` | Parent–child | child Chroma + parent map | child hit → parent expand |
| `proposition_rag` | Proposition index | propositions.json + Chroma | dense over atomic claims |

**Important GraphRAG nuance:** indexing cost (`fast` NLP vs `standard` LLM extract) is separate from search flavor (`global` / `local` / `basic` / `drift`). `graph_rag` and `graph_local_rag` share one index and only differ at query time.

`lazygraph_rag` is **not** Microsoft LazyGraphRAG (not OSS here). It is GraphRAG with **fast** indexing + **basic** search in its own workspace.


### 3a. Semantic / dense-sparse / rerank

**Code:** `src/rag_benchmark/semantic_rag.py`, `modern_rag.py`, `sdk/vector_store.py`

```
corpus_*.txt
   → load_documents → chunk_documents
   → embed each chunk (e.g. all-MiniLM-L6-v2)
   → upsert into Chroma under .chroma/sdk_lineage/{collection}__{embedder}/
```

| Variant | Extra at index | Extra at query |
|---------|----------------|----------------|
| Semantic | Chroma only | embed Q → top-k → generate |
| BM25+dense | + `BM25Okapi` over chunk texts | RRF of dense + sparse ranks |
| Vector + rerank | same Chroma | fetch `top_k*4`, CrossEncoder re-rank |

Metadata on each vector includes lineage (`source_id`, `content_hash`, `chunk_id`) so incremental sync can skip unchanged docs when `reuse_indexes=True`.


In [6]:
chroma_root = ROOT / ".chroma"
lines = ["### Chroma / lineage dirs present"]
if chroma_root.exists():
    for p in sorted(chroma_root.rglob("*")):
        if p.is_dir() and p != chroma_root:
            # only show shallow interesting folders
            rel = p.relative_to(ROOT)
            if len(rel.parts) <= 4:
                lines.append(f"- `{rel}/`")
else:
    lines.append("_No `.chroma/` yet — run a semantic method once._")
display(Markdown("\n".join(lines[:40])))

### Chroma / lineage dirs present
- `.chroma/embedding_bakeoff/`
- `.chroma/embedding_bakeoff/embed_bakeoff__all_minilm_l6_v2/`
- `.chroma/embedding_bakeoff/embed_bakeoff__all_minilm_l6_v2/7da5aefd-34a1-4713-aa20-812af1cd6ee7/`
- `.chroma/graphrag_bench_semantic/`
- `.chroma/graphrag_bench_semantic/41f267dc-985f-408b-a004-974a49726a8d/`
- `.chroma/hotpot_semantic/`
- `.chroma/hotpot_semantic/81641033-7d7d-4925-bc94-c3cc76910fcc/`
- `.chroma/multihop_semantic/`
- `.chroma/multihop_semantic/a3f1c49e-0857-4a02-9590-a906689ab805/`
- `.chroma/semantic_rag_corpus/`
- `.chroma/semantic_rag_corpus/18ea9be7-ca1d-41b7-91c2-fbf5440ad59b/`
- `.chroma/semantic_rag_corpus/d343aa6b-815d-4893-b862-97573dfa44dd/`
- `.chroma/smoke_prop/`
- `.chroma/smoke_prop/1143995f-df7c-46bf-b385-4536e110c2d2/`

### 3b. GraphRAG family (global / local / DRIFT / fast+basic)

**Code:** `src/rag_benchmark/graph_rag.py`

```
corpus_*.txt
   → copy into {workspace}/input/
   → graphrag init  (settings.yaml, prompts)
   → graphrag index --method fast|standard
   → output/*.parquet
```

**Index methods**

| `--method` | Entity extract | Notes |
|------------|----------------|-------|
| `fast` | spaCy NLP noun phrases (`en_core_web_sm`) | Default for local 3B; cheap |
| `standard` | LLM entity extraction | Needs stronger models |

**Search methods** (same parquet index, different query CLI)

| Method key | Workspace | Search |
|------------|-----------|--------|
| `graph_rag` | `graphrag_workspaces/hotpot/` | global (community reports) |
| `graph_local_rag` | same | local (entity neighborhood) |
| `drift_rag` | same | drift |
| `lazygraph_rag` | `graphrag_workspaces/hotpot_lazy/` | **basic** + forced **fast** index |

Typical `output/` artifacts:

```
documents.parquet      # ingested docs
text_units.parquet     # GraphRAG text units
entities.parquet       # nodes
relationships.parquet  # edges
communities.parquet
community_reports.parquet   # needed for global search
```


In [7]:
def describe_workspace(label: str, root: Path) -> pd.DataFrame:
    out = root / "output"
    rows = []
    if not root.exists():
        return pd.DataFrame([{"workspace": label, "path": str(root.relative_to(ROOT)), "status": "missing"}])
    input_n = len(list((root / "input").glob("*.txt"))) if (root / "input").exists() else 0
    rows.append({"workspace": label, "artifact": "input/*.txt", "detail": f"{input_n} files"})
    if out.exists():
        for p in sorted(out.iterdir()):
            if p.is_file():
                detail = f"{p.stat().st_size // 1024} KB"
                if p.suffix == ".parquet":
                    try:
                        detail += f", {len(pd.read_parquet(p))} rows"
                    except Exception as e:  # noqa: BLE001
                        detail += f" (read err: {e})"
                rows.append({"workspace": label, "artifact": p.name, "detail": detail})
    else:
        rows.append({"workspace": label, "artifact": "output/", "detail": "missing — not indexed"})
    return pd.DataFrame(rows)

workspaces = [
    ("GraphRAG full/hotpot (global+local)", ROOT / "graphrag_workspaces" / "hotpot"),
    ("GraphRAG fast/basic (lazy workspace)", ROOT / "graphrag_workspaces" / "hotpot_lazy"),
]
frames = [describe_workspace(label, path) for label, path in workspaces]
display(Markdown("#### GraphRAG workspace artifacts on disk"))
display(pd.concat(frames, ignore_index=True))

entities = ROOT / "graphrag_workspaces" / "hotpot" / "output" / "entities.parquet"
if entities.exists():
    edf = pd.read_parquet(entities)
    display(Markdown("#### Sample entities (hotpot GraphRAG index)"))
    cols = [c for c in ["title", "type", "description", "human_readable_id"] if c in edf.columns]
    display(edf[cols].head(8) if cols else edf.head(8))

#### GraphRAG workspace artifacts on disk

,workspace,artifact,detail
0,GraphRAG full/hotpot (global+local),input/*.txt,119 files
1,GraphRAG full/hotpot (global+local),communities.parquet,"14 KB, 2 rows"
2,GraphRAG full/hotpot (global+local),community_reports.parquet,"37 KB, 2 rows"
3,GraphRAG full/hotpot (global+local),context.json,0 KB
4,GraphRAG full/hotpot (global+local),documents.parquet,"86 KB, 119 rows"
5,GraphRAG full/hotpot (global+local),entities.parquet,"25 KB, 99 rows"
6,GraphRAG full/hotpot (global+local),relationships.parquet,"26 KB, 260 rows"
7,GraphRAG full/hotpot (global+local),stats.json,2 KB
8,GraphRAG full/hotpot (global+local),text_units.parquet,"102 KB, 120 rows"
9,GraphRAG fast/basic (lazy workspace),input/*.txt,119 files


#### Sample entities (hotpot GraphRAG index)

,title,type,description,human_readable_id
0,ERSKINE HAMILTON CHILDERS,NOUN PHRASE,,0
1,GRETCHEN OSGOOD WARREN,NOUN PHRASE,,1
2,ROBERT ERSKINE CHILDERS,NOUN PHRASE,,2
3,IRISH CIVIL WAR,NOUN PHRASE,,3
4,MARY ALDEN CHILDERS,NOUN PHRASE,,4
5,ROBERT BARTON,NOUN PHRASE,,5
6,KOREAN,NOUN PHRASE,,6
7,CALIFORNIA,NOUN PHRASE,,7


### 3c. Hybrid, Adaptive, Frontier

**Hybrid (`hybrid_rag`)** — `hybrid_rag.py`

1. Index = semantic Chroma **and** GraphRAG workspace (reuses main graph index; search = **local**)
2. Query = retrieve vector chunks (no separate semantic answer) + GraphRAG local prose → **one** fusion LLM call

**Adaptive (`adaptive_rag`)** — rule-classifies question → calls semantic *or* hybrid (no own store beyond those two).

**Frontier (`frontier_rag`)** — indexes BM25+dense (+ shared rerank store) and hybrid; at query: wide retrieve → cross-encoder → YES/NO grade → escalate to hybrid on multi-hop / failed grade (CRAG-style).


### 3d. LightRAG & HippoRAG

**LightRAG** (`light_rag.py`, HKUDS) — inserts full docs into a LightRAG workspace:

```
lightrag_workspaces/hotpot/
  kv_store_full_docs.json
  kv_store_text_chunks.json
  vdb_chunks.json / vdb_entities.json / vdb_relationships.json
  graph_chunk_entity_relation.graphml
```

Query modes: `naive | local | global | hybrid | mix` (config default `hybrid`).

**HippoRAG 2** (`hippo_rag.py`) — OpenIE extraction + synonymy graph:

```
hipporag_workspaces/hotpot/
  openie_results_ner_*.json
  */graph.pickle
  llm_cache/*.sqlite
```

Indexing is LLM-heavy (slow on 3B). Opt-in via `scripts/run_hotpot_new_methods.py`.


In [8]:
def list_dir(label: str, path: Path, limit: int = 20) -> None:
    display(Markdown(f"#### {label}"))
    if not path.exists():
        display(Markdown(f"_Missing `{path.relative_to(ROOT)}`_"))
        return
    files = sorted([p for p in path.rglob("*") if p.is_file()])[:limit]
    display(pd.DataFrame(
        [{"file": str(p.relative_to(path)), "KB": p.stat().st_size // 1024} for p in files]
    ))

list_dir("LightRAG hotpot workspace", ROOT / "lightrag_workspaces" / "hotpot")
list_dir("HippoRAG hotpot workspace", ROOT / "hipporag_workspaces" / "hotpot")

#### LightRAG hotpot workspace

,file,KB
0,graph_chunk_entity_relation.graphml,308
1,kv_store_doc_status.json,205
2,kv_store_entity_chunks.json,103
3,kv_store_full_docs.json,127
4,kv_store_full_entities.json,27
5,kv_store_full_relations.json,28
6,kv_store_llm_response_cache.json,2081
7,kv_store_relation_chunks.json,65
8,kv_store_text_chunks.json,95
9,vdb_chunks.json,625


#### HippoRAG hotpot workspace

,file,KB
0,llama3.2:3b_nomic-embed-text/chunk_embeddings/vdb_chunk.parquet,956
1,llama3.2:3b_nomic-embed-text/entity_embeddings/vdb_entity.parquet,6941
2,llama3.2:3b_nomic-embed-text/fact_embeddings/vdb_fact.parquet,6967
3,llama3.2:3b_nomic-embed-text/graph.pickle,311
4,llm_cache/llama3.2:3b_cache.sqlite,248
5,llm_cache/llama3.2:3b_cache.sqlite.lock,0
6,openie_results_ner_llama3.2:3b.json,165


### 3e. Memory structures (RAPTOR / parent–child / propositions)

**Code:** `src/rag_benchmark/memory_structures.py` — workspaces under `memory_workspaces/`.

| Method | Index steps | Artifacts | Retrieve |
|--------|-------------|-----------|----------|
| **RAPTOR** | chunk → embed → GMM cluster → LLM summarize → recurse | `tree.json` + Chroma of all levels | collapsed tree: top-k across levels |
| **Parent–child** | large parent chunks → small child chunks | `parent_child_map.json` + child-only Chroma | hit child → expand unique parents |
| **Proposition** | chunk → LLM atomic claims | `propositions.json` + Chroma | dense over propositions |

These reshape *text granularity* rather than building a knowledge graph.


In [9]:
mem = ROOT / "memory_workspaces"
if mem.exists():
    rows = []
    for p in sorted(mem.rglob("*")):
        if p.is_file() and p.suffix in {".json", ".parquet"}:
            rows.append({"path": str(p.relative_to(ROOT)), "KB": p.stat().st_size // 1024})
    display(Markdown("#### Memory-structure artifacts"))
    display(pd.DataFrame(rows) if rows else Markdown("_No tree/map/proposition JSON yet (smoke dirs may be empty)._"))
else:
    display(Markdown("_No `memory_workspaces/` directory._"))

#### Memory-structure artifacts

,path,KB
0,memory_workspaces/_smoke2_pc/parent_child_map.json,81
1,memory_workspaces/_smoke2_prop/propositions.json,75
2,memory_workspaces/_smoke2_raptor/tree.json,36
3,memory_workspaces/_smoke_prop/propositions.json,102
4,memory_workspaces/_smoke_raptor/tree.json,48
5,memory_workspaces/_smoke_tiered/doc_summaries.json,1


## 4. End-to-end picture (Hotpot bake-off)

```
hotpotqa/hotpot_qa (HF distractor)
        │  build_hotpot_subset(n=24)
        ▼
data/corpus_hotpot/*.txt          data/qa/hotpot_eval.json
        │
        ├─► Semantic / rerank / BM25+dense  →  .chroma/sdk_lineage/hotpot_semantic__…
        ├─► GraphRAG global/local/hybrid   →  graphrag_workspaces/hotpot/output/*.parquet
        ├─► GraphRAG fast/basic            →  graphrag_workspaces/hotpot_lazy/output/*.parquet
        ├─► LightRAG (opt-in)              →  lightrag_workspaces/hotpot/
        └─► HippoRAG (opt-in)              →  hipporag_workspaces/hotpot/
                │
                ▼
        BenchmarkRunner evaluates each method on hotpot_eval.json
                │
                ▼
        results/accuracy_results.csv  (+ autopsy notebook)
```

### Rebuild tips

```bash
# Expand / refresh Hotpot corpus + run core methods (rebuilds indexes when n>12)
python scripts/run_hotpot_benchmark.py 24

# GraphRAG-Bench Novel subset
python scripts/run_graphrag_bench.py

# MultiHop-RAG mini subset
python scripts/run_multihop_benchmark.py
```

When the corpus gains docs, set `reuse_indexes=False` (the Hotpot script does this automatically for `n>12`) so Chroma/GraphRAG are not stale relative to `data/corpus_*`.


In [10]:
cheat = pd.DataFrame(
    [
        {"method": k, "label": v}
        for k, v in METHOD_LABELS.items()
    ]
)
cheat["index_kind"] = [
    "Chroma chunks",
    "GraphRAG parquet (global query)",
    "GraphRAG parquet (local query)",
    "Chroma + GraphRAG local fuse",
    "GraphRAG parquet (drift query)",
    "GraphRAG fast+basic (own workspace)",
    "LightRAG kv/vdb/graphml",
    "HippoRAG OpenIE graph",
    "Chroma + BM25",
    "Chroma + CrossEncoder",
    "Router over semantic/hybrid",
    "BM25+dense + CRAG escalate",
    "RAPTOR tree + Chroma",
    "Child Chroma → parent expand",
    "Proposition Chroma",
]
display(Markdown("### Quick reference"))
display(cheat)

### Quick reference

,method,label,index_kind
0,semantic_rag,Semantic (vector),Chroma chunks
1,graph_rag,GraphRAG global,GraphRAG parquet (global query)
2,graph_local_rag,GraphRAG local,GraphRAG parquet (local query)
3,hybrid_rag,Hybrid (vec+graph local),Chroma + GraphRAG local fuse
4,drift_rag,DRIFT (GraphRAG hybrid),GraphRAG parquet (drift query)
5,lazygraph_rag,GraphRAG fast/basic,GraphRAG fast+basic (own workspace)
6,light_rag,LightRAG (HKUDS),LightRAG kv/vdb/graphml
7,hippo_rag,HippoRAG 2,HippoRAG OpenIE graph
8,hybrid_dense_sparse,BM25+dense (RRF),Chroma + BM25
9,rerank_semantic,Vector + rerank,Chroma + CrossEncoder
